# LangGraph MCP Tool-Calling Agent - Interactive Testing

This notebook demonstrates how to test and deploy a LangGraph agent that connects to MCP servers hosted on Databricks.

## Features
- Author a LangGraph agent wrapped with `ResponsesAgent` for Mosaic AI compatibility
- Call MCP tools (managed or custom)
- Manually test the agent
- Evaluate with Mosaic AI Agent Evaluation
- Log and deploy the agent

In [ ]:
# Install required packages
%pip install -U -qqqq langgraph uv databricks-agents mlflow-skinny[databricks] databricks-mcp databricks-langchain azure-identity azure-search-documents

## Step 1: Test the Partition Planner

The graph runs in two turns:
- **Turn 1** – planner decomposes the query and hits `interrupt()`, returning proposed sub-topics
- **Turn 2** – user confirms (or edits) the list, workers run in parallel, synthesizer produces the final answer

# COMMAND ----------

## Step 0: Local Authentication Setup (for running outside Databricks)

When running this notebook locally, you need to authenticate with Databricks. You have two options:

### Option 1: Databricks CLI OAuth Login (Recommended for local development)
```bash
# Install Databricks CLI if not already installed
pip install databricks-cli

# Login using OAuth (opens browser for authentication)
databricks auth login --host https://your-workspace.cloud.databricks.com
```

This creates a configuration profile that the SDK will automatically use.

### Option 2: Environment Variables with OAuth M2M
For automated/CI environments, use OAuth machine-to-machine with a service principal:
```bash
export DATABRICKS_HOST="https://your-workspace.cloud.databricks.com"
export DATABRICKS_CLIENT_ID="your-service-principal-client-id"
export DATABRICKS_CLIENT_SECRET="your-service-principal-secret"
```

The code below will automatically detect and use whichever authentication method is available.

In [2]:
# COMMAND ----------

# Test Databricks authentication
from databricks.sdk import WorkspaceClient
import os

# Use the profile name (for local development)
profile_name = os.getenv("DATABRICKS_CONFIG_PROFILE", "development")

try:
    ws = WorkspaceClient(profile=profile_name)
    print(f"✓ Successfully authenticated to: {ws.config.host}")
    print(f"✓ Using profile: {profile_name}")
    print(f"✓ Auth type: {ws.config.auth_type}")
except Exception as e:
    print(f"✗ Authentication failed: {e}")
    print("\nTo fix, run: databricks auth login --host https://your-workspace.cloud.databricks.com")

✓ Successfully authenticated to: https://adb-3253299566947192.12.azuredatabricks.net
✓ Using profile: development
✓ Auth type: metadata-service


# COMMAND ----------

## Step 2: Test the Agent

Load and test the agent with sample queries.

In [4]:
# COMMAND ----------

# Configure MLflow tracking BEFORE importing agent
import mlflow
import os

profile_name = os.getenv("DATABRICKS_CONFIG_PROFILE", "development")

try:
    # Set tracking URI with profile
    mlflow.set_tracking_uri(f"databricks://{profile_name}")
    print(f"✓ MLflow tracking configured: databricks://{profile_name}")
    
    # Set experiment name
    experiment_name = "/Users/huy.d@hotmail.com/langgraph-mcp-agent"  # TODO: Update with your email
    mlflow.set_experiment(experiment_name)
    print(f"✓ Experiment set to: {experiment_name}")
except Exception as e:
    print(f"Note: MLflow tracking not configured: {e}")
    print("Agent will work without MLflow tracking.")

✓ MLflow tracking configured: databricks://development
✓ Experiment set to: /Users/huy.d@hotmail.com/langgraph-mcp-agent
✓ Experiment set to: /Users/huy.d@hotmail.com/langgraph-mcp-agent


In [ ]:
import uuid
from agent import partition_graph
from langgraph.types import Command

# Each conversation needs a stable thread_id so the checkpointer can resume after interrupt
thread_id = str(uuid.uuid4())
thread_config = {"configurable": {"thread_id": thread_id}}

# Turn 1: run until the HITL interrupt fires after the planner
result = partition_graph.invoke(
    {
        "query": "What are the best practices for data engineering pipelines?",
        "partitions": [],
        "research_results": [],
        "final_answer": "",
    },
    config=thread_config,
)

if "__interrupt__" in result:
    interrupt_data = result["__interrupt__"][0].value
    print(interrupt_data["message"])
    print()
    for i, p in enumerate(interrupt_data["partitions"], 1):
        print(f"  {i}. {p}")
else:
    print("No interrupt — something went wrong with graph wiring")

In [ ]:
# Turn 2: resume with the approved partitions — workers run in parallel, synthesizer produces final answer
# Edit interrupt_data["partitions"] here if you want to change the list before resuming

final = partition_graph.invoke(
    Command(resume={"partitions": interrupt_data["partitions"]}),
    config=thread_config,
)

print("=== Final Answer ===")
print(final["final_answer"])
print()
print(f"Workers ran: {len(final['research_results'])} parallel searches")
assert len(final["research_results"]) == len(interrupt_data["partitions"]), "result count should match partition count"

# COMMAND ----------

## Step 3: Evaluate the Agent

Run evaluation with predefined LLM scorers.

In [7]:
# COMMAND ----------

import mlflow
from mlflow.genai.scorers import RelevanceToQuery, Safety

eval_dataset = [
    {
        "inputs": {
            "input": [
                {
                    "role": "user",
                    "content": "Calculate the 15th Fibonacci number"
                }
            ]
        },
        "expected_response": "The 15th Fibonacci number is 610."
    }
]

eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=lambda input: AGENT.predict({"input": input}),
    scorers=[RelevanceToQuery(), Safety()],
)

# Review the evaluation results in the MLflow UI
print("Evaluation complete. Check the MLflow UI for detailed results.")

2025/11/08 22:44:33 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/11/08 22:44:33 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2025/11/08 22:44:33 WARNING mlflow.tracing.fluent: Failed to start span predict_stream: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2025/11/08 22:44:33 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NonRecordingSpan' object has no attribute 'context'. For full traceback, set logging level to debug.
2025/11/08 22:44:33 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_S

Evaluation complete. Check the MLflow UI for detailed results.


# COMMAND ----------

## Step 4: Log the Agent as an MLflow Model

Log the agent code for deployment.

In [ ]:
import mlflow
import os
from agent import LLM_ENDPOINT_NAME, AZURE_SEARCH_ENDPOINT, AZURE_SEARCH_INDEX
from mlflow.models.resources import DatabricksServingEndpoint
from pkg_resources import get_distribution

profile_name = os.getenv("DATABRICKS_CONFIG_PROFILE", "development")

try:
    mlflow.set_tracking_uri(f"databricks://{profile_name}")
    experiment_name = "/Users/huy.d@hotmail.com/langgraph-mcp-agent"
    mlflow.set_experiment(experiment_name)
    print(f"✓ MLflow tracking configured with profile: {profile_name}")
except Exception as e:
    print(f"Warning: Could not set tracking URI: {e}")

resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
]

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            "azure-search-documents>=11.4.0",
            "azure-identity>=1.15.0",
            "databricks-mcp",
            f"langgraph=={get_distribution('langgraph').version}",
            f"mcp=={get_distribution('mcp').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
        ]
    )

print(f"Model logged: {logged_agent_info.model_uri}")

# COMMAND ----------

## Step 5: Pre-deployment Validation

Validate the model before deploying.

In [9]:
# COMMAND ----------

mlflow.models.predict(
    model_uri=f"runs:/{logged_agent_info.run_id}/agent",
    input_data={"input": [{"role": "user", "content": "What is 7*6 in Python?"}]},
    env_manager="uv",
)


2025/11/08 22:45:05 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/11/08 22:45:05 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'

2025/11/08 22:45:08 INFO mlflow.utils.virtualenv: Environment /tmp/virtualenv_envs/mlflow-fe38911592d12d9b07553a1cb2d44173ca6747d4 already exists
2025/11/08 22:45:08 WARNING mlflow.utils.databricks_utils: Missing required environment variable `SPARK_LOCAL_REMOTE` or `SPARK_REMOTE`. These are necessary to initialize the WorkspaceClient with serverless compute in a subprocess in Databricks for UC function execution. Setting the value to 'true'.
2025/11/08 22:45:08 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-fe38911592d12d9b07553a1cb2d44173ca6747d4/bin/activate && python -c ""']'

2025/11/08 22:45:08 INFO mlflow.utils.virtualenv: Environment /tmp/virtualenv_envs/mlflow-fe38911592d12d9b07553a1cb2d44173ca6747d4 alr

✓ MLflow autologging enabled
{"object": "response", "output": [{"type": "message", "id": "lc_run--80ee1d12-6c02-42ec-97c8-0bc4423ecade", "content": [{"text": "I can calculate 7*6 using Python. Let me run the code for you:", "type": "output_text"}], "role": "assistant"}, {"type": "function_call", "id": "lc_run--80ee1d12-6c02-42ec-97c8-0bc4423ecade", "call_id": "toolu_bdrk_01UYADGEiB1ForPtNhJ3fcZN", "name": "system__ai__python_exec", "arguments": "{\"code\": \"print(7 * 6)\"}"}, {"type": "function_call_output", "call_id": "toolu_bdrk_01UYADGEiB1ForPtNhJ3fcZN", "output": "{\"is_truncated\":false,\"columns\":[\"output\"],\"rows\":[[\"42\\n\"]]}"}, {"type": "message", "id": "lc_run--24ace0f4-336c-4482-8d1d-45c8acbc0c0d", "content": [{"text": "The result of 7*6 in Python is 42.", "type": "output_text"}], "role": "assistant"}]}{"object": "response", "output": [{"type": "message", "id": "lc_run--80ee1d12-6c02-42ec-97c8-0bc4423ecade", "content": [{"text": "I can calculate 7*6 using Python. Let 

2025/11/08 22:45:15 INFO mlflow.tracing.export.async_export_queue: Flushing the async trace logging queue before program exit. This may take a while...


# COMMAND ----------

## Step 6: Register to Unity Catalog

Register the agent to Unity Catalog before deployment.

In [10]:
# COMMAND ----------

import os

# Set registry URI with profile (same format as tracking URI)
profile_name = os.getenv("DATABRICKS_CONFIG_PROFILE", "development")
mlflow.set_registry_uri(f"databricks-uc://{profile_name}")
print(f"✓ Registry URI set to: databricks-uc://{profile_name}")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "rag"
schema = "development"
model_name = "langgraph_mcp_agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

# Register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, 
    name=UC_MODEL_NAME
)

print(f"Model registered: {UC_MODEL_NAME} (version {uc_registered_model_info.version})")

✓ Registry URI set to: databricks-uc://development


Registered model 'rag.development.langgraph_mcp_agent' already exists. Creating a new version of this model...

Uploading artifacts: 100%|██████████| 10/10 [00:00<00:00, 11.10it/s]

🔗 Created version '3' of model 'rag.development.langgraph_mcp_agent': https://adb-3253299566947192.12.azuredatabricks.net/explore/data/models/rag/development/langgraph_mcp_agent/version/3
🔗 Created version '3' of model 'rag.development.langgraph_mcp_agent': https://adb-3253299566947192.12.azuredatabricks.net/explore/data/models/rag/development/langgraph_mcp_agent/version/3


Model registered: rag.development.langgraph_mcp_agent (version 3)


# COMMAND ----------

## Step 7: Deploy the Agent

Deploy the agent to a model serving endpoint.

In [ ]:
# COMMAND ----------

from databricks import agents

deployment_info = agents.deploy(
    UC_MODEL_NAME, 
    uc_registered_model_info.version,
    scale_to_zero_enabled=True,
    tags={"endpointSource": "langgraph_mcp_notebook"}
)

print(f"Agent deployed successfully!")
print(f"Endpoint: {deployment_info}")


2025/11/08 22:45:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/08 22:45:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-3253299566947192.12.azuredatabricks.net/ml/experiments/897801252548174/models/m-530aa6d09e834f0a8f62df2744791055
🔗 View Logged Model at: https://adb-3253299566947192.12.azuredatabricks.net/ml/experiments/897801252548174/models/m-530aa6d09e834f0a8f62df2744791055
2025/11/08 22:45:25 INFO mlflow.pyfunc: Validating input example against model signature
2025/11/08 22:45:25 INFO mlflow.pyfunc: Validating input example against model signature
Successfully registered model 'rag.development.feedback'.
Successfully registered model 'rag.development.feedback'.
Uploading artifacts: 100%|██████████| 8/8 [00:00<00:00, 19.98it/s]

🔗 Created version '1' of model 'rag.development.feedback': https://adb-3253299566947192.12.azuredatabricks.net/explore/data/m

🏃 View run feedback-model at: https://adb-3253299566947192.12.azuredatabricks.net/ml/experiments/897801252548174/runs/41a7418d740a4bebaf00dfdeb3c1c693
🧪 View experiment at: https://adb-3253299566947192.12.azuredatabricks.net/ml/experiments/897801252548174

    Deployment of rag.development.langgraph_mcp_agent version 3 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://adb-3253299566947192.12.azuredatabricks.net/ml/endpoints/agents_rag-development-langgraph_mcp_agent/
    Review App: https://adb-3253299566947192.12.azuredatabricks.net/ml/review-v2/34911c7cde294e0bbfafd6c85f583473/chat
    Monitor: https://adb-3253299566947192.12.azuredatabricks.net/ml/experiments/897801252548174?compareRunsMode=TRACES

You can refer back to the links above from the endpoint detail page at https://adb-3253299566947192.12.azuredatabricks.net/ml/endpoints/agents_rag-development-langgraph_mcp_agent/.
Agent d

# COMMAND ----------

## Next Steps

After your agent is deployed, you can:
- Chat with it in AI Playground
- Share it with SMEs for feedback
- Embed it in a production application
- Monitor its performance and traces